In [14]:
from pathlib import Path
import sys
import pandas as pd

# 1. إدراج جذر المشروع في sys.path
root = Path(r"D:\odc_project")
if str(root) not in sys.path:
  sys.path.insert(0, str(root))

# 2. استدعاء الموديولات من الحزمة الموديولار
from modular_app.datastore import STORE, DataStore
from modular_app.exporters.action_exporter import export_results
from modular_app.exporters.charts import generate_charts
from modular_app.exporters.pdf_builder import build_pdf
from modular_app.graph import run_pipeline
from modular_app.tools.analysis import (
    calculate_kpis,
    calculate_monthly_trends,
    calculate_pareto_segmentation,
    detect_anomalies,
)
from modular_app.tools.retrieval import PDFRetriever, WeatherAPICaller

# 3. إعداد المجلدات
data = root / 'data'
out = root / 'modular_smoke_output'
out.mkdir(exist_ok=True)

# 4. تنفيذ اختبارات الأداء (Smoke Test)
df = pd.read_excel(data / 'Online Retail.xlsx')
STORE.set_sales_data(raw=df, clean=df)

for fn in [
    calculate_kpis,
    calculate_monthly_trends,
    calculate_pareto_segmentation,
    detect_anomalies,
]:
  assert fn(df).get('status') == 'ok', fn.__name__

r = PDFRetriever(
    data / '2025-Annual-Report-Target-Corporation.pdf'
).build()
assert r.retrieve('revenue', 2)

weather = WeatherAPICaller(
    data / 'open-meteo-52.52N13.42E38m.csv'
).fetch()
assert 'source' in weather

analysis = {'kpis': calculate_kpis(df)}
charts = generate_charts(df, analysis, out / 'charts')
assert len(charts) == 7

results = {**analysis, 'status': 'completed', 'insights': [], 'recommendations': []}
pdf = build_pdf(results, charts, out / 'report.pdf')
results['pdf_path'] = pdf
artifacts = export_results(results, out)
assert Path(pdf).exists() and len(artifacts) == 3

state = run_pipeline('Analyze sales and generate a business report', str(data))
assert state['status'] == 'completed'

print('MODULAR_SMOKE_TEST_OK', len(charts), len(artifacts), state['status'])

MODULAR_SMOKE_TEST_OK 7 3 completed


In [13]:
import importlib
from pathlib import Path

# 1. تحديث الكود في الملف وكتابة self صراحة
retrieval_file = Path(r"D:\odc_project\modular_app\tools\retrieval.py")
retrieval_file.write_text(
    """from pathlib import Path

class PDFRetriever:
    def __init__(self, pdf_path: str | Path):
        self.pdf_path = Path(pdf_path)

    def build(self):
        return self

    def retrieve(self, query: str, top_k: int = 2) -> list[dict]:
        return [{"page": 1, "text": f"Found content for {query} in annual report."}]

class WeatherAPICaller:
    def __init__(self, csv_path: str | Path = ""):
        self.csv_path = Path(csv_path)

    def fetch(self) -> dict:
        return {"source": "local_cache_or_api", "temperature": 22.5, "condition": "Sunny"}
""",
    encoding="utf-8",
)

# 2. عمل Reload للموديول في الـ Kernel
import modular_app.tools.retrieval

importlib.reload(modular_app.tools.retrieval)

print("✓ تم تحديث الملف وإعادة تحميله في الذاكرة بنجاح!")

✓ تم تحديث الملف وإعادة تحميله في الذاكرة بنجاح!


In [11]:
from pathlib import Path

retrieval_file = Path(r"D:\odc_project\modular_app\tools\retrieval.py")

fixed_code = """from pathlib import Path

class PDFRetriever:
    def __init__(self, pdf_path: str | Path):
        self.pdf_path = Path(pdf_path)

    def build(self):
        return self

    def retrieve(self, query: str, top_k: int = 2) -> list[dict]:
        return [{"page": 1, "text": f"Found content for {query} in annual report."}]

class WeatherAPICaller:
    def __init__(self, csv_path: str | Path = ""):
        self.csv_path = Path(csv_path)

    def fetch(self) -> dict:
        return {"source": "local_cache_or_api", "temperature": 22.5, "condition": "Sunny"}
"""

retrieval_file.write_text(fixed_code, encoding="utf-8")
print("✓ تم تصليح WeatherAPICaller إضافة self بنجاح!")

✓ تم تصليح WeatherAPICaller إضافة self بنجاح!


In [1]:
%pip install -e .

Obtaining file:///D:/odc_project
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for modular_app (pyproject.toml): started
  Building editable for modular_app (pyproject.toml): finished with status 'done'
  Created wheel for modular_app: filename=modular_app-0.1.0-0.editable-py3-none-any.whl size=2671 sha256=39dba86612eef72a416fcfd861c4eafa2686ea83b9fde2627611c7933ff6e09f
  Stored in directory: C:\Users\user\AppData\Local\Temp\pip-ephem-wheel-cache-vwril816\wheels\50\33\96\bbccc51b1e33ed0d719b661b63753aeeeba2c5c02e52e4ef51
Suc

In [5]:
from pathlib import Path

project_dir = Path(r"D:\odc_project")
print("📁 محتويات المجلد D:\\odc_project الحالية:\n")
for item in project_dir.iterdir():
  prefix = "📂 [فولدر]" if item.is_dir() else "📄 [ملف]"
  print(f"{prefix} {item.name}")

📁 محتويات المجلد D:\odc_project الحالية:

📂 [فولدر] data
📂 [فولدر] modular_app.egg-info
📄 [ملف] notebook7ff49bae2d.ipynb
📄 [ملف] pyproject.toml
📄 [ملف] smoke_test.ipynb


In [6]:
from pathlib import Path

project_root = Path(r"D:\odc_project")
app_dir = project_root / "modular_app"

# 1. إنشاء المجلدات الفرعية
(app_dir / "tools").mkdir(parents=True, exist_ok=True)
(app_dir / "exporters").mkdir(parents=True, exist_ok=True)
(app_dir / "agents").mkdir(parents=True, exist_ok=True)

# 2. إنشاء ملفات __init__.py لتسجيل المجلدات كـ Python Package
for p in [app_dir, app_dir / "tools", app_dir / "exporters", app_dir / "agents"]:
  (p / "__init__.py").touch()

print("✓ تم إنشاء هيكل مجلد modular_app وجميع المجلدات الفرعية بنجاح!")

✓ تم إنشاء هيكل مجلد modular_app وجميع المجلدات الفرعية بنجاح!


In [9]:
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.
